In [ ]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

from data_processing_utils import extract_yearly_defaults

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# Partitioning SFLLD

In [ ]:
years = [str(i) for i in range(1999, 2024 + 1)]
annual_counts = {}
for y in years:
    N, orig_df, svcg_df, lgd_df = extract_yearly_defaults(y)
    orig_df.to_feather(f"data/default_data/{y}/default_orig.feather")
    svcg_df.to_feather(f"data/default_data/{y}/default_svcg.feather")
    lgd_df.to_feather(f"data/default_data/{y}/default_lgd.feather")

    annual_counts[y] = N

# Processing SFLLD

In [ ]:
full_orig_df, full_lgd_df = pd.DataFrame(), pd.DataFrame()

years = [str(i) for i in range(1999, 2024 + 1)]
for y in years:
    y_orig = pd.read_feather(f"data/default_data/{y}/default_orig.feather")
    y_lgd = pd.read_feather(f"data/default_data/{y}/default_lgd.feather")

    full_orig_df = pd.concat([full_orig_df, y_orig], ignore_index=True)
    full_lgd_df = pd.concat([full_lgd_df, y_lgd], ignore_index=True)

# For analysis we use "LGD2"
full_lgd_df['uncensored_LGD'] = full_lgd_df['LGD2']
full_lgd_df['LGD'] = np.minimum(np.maximum(full_lgd_df['LGD2'], 0), 1)

full_lgd_df = full_lgd_df.drop(['LGD0', 'LGD1', 'LGD2', 'loss_severity'], axis=1)

# Use only observations defaulting between 2000 and 2022 (inclusive)
lgd_df_clipped = full_lgd_df[(full_lgd_df['date_default'].dt.year >= 2000) & (full_lgd_df['date_default'].dt.year <= 2022)]
orig_df_clipped = full_orig_df[full_orig_df['loan_number'].isin(lgd_df_clipped['loan_number'])]

orig_df_clipped.to_feather("data/processed_data/clipped_data_origination.feather")
lgd_df_clipped.to_feather("data/processed_data/clipped_data_loss.feather")

In [ ]:
# Importing data
macro_dtypes = {
    "year": "string",
    "state_long": "string",
    "state_short": "string",
    "gdp_growth": "float",
    "ln_income_per_capita": "float",
    "ln_expenditures_per_capita": "float",
    "unemployment_rate": "float",
    "hpi_growth": "float",
    "gdp_construction_growth": "float",
    "gross_operating_surplus_growth": "float",
    "inflation_rate": "float",
    "DJIA_growth": "float",
}
rate_dtypes = {
    "observation_date": "string",
    "MORTGAGE30US": "float"
}

# Mortgage data
orig_df = pd.read_feather("data/processed_data/clipped_data_origination.feather")
lgd_df = pd.read_feather("data/processed_data/clipped_data_loss.feather")
# Adding year_default to LGD dataframe
lgd_df['year_default'] = lgd_df['date_default'].dt.year.astype(np.float64)

# Macro data
macro_df = pd.read_csv("data/supplemental_feature_data/macro_data.csv", sep=";", dtype=macro_dtypes)
# Avg mortgage rate
rate_df = pd.read_csv("data/supplemental_feature_data/MORTGAGE30US.csv", sep=',', dtype=rate_dtypes)

In [ ]:
orig_vars = ["loan_number", "date_loan_start", "X_centroid", "Y_centroid", 'ZIP3', "property_state", "credit_score", "original_interest_rate", "occupancy", "nr_units", "loan_purpose",
                "first_time_homebuyer", "MSA", "insurance_percent", "original_debt_to_income", "original_combined_loan_to_value", "original_loan_to_value", "original_upb",
                "number_of_borrowers"]
loss_vars = ["loan_number", "date_default", 'year_default', "upb_at_default", "LGD"]
# TODO: engineer performance-based features

variables_df = orig_df[orig_vars].merge(lgd_df[loss_vars], on="loan_number", how='left')
variables_df = variables_df.rename(columns={"property_state": "state_short"})

# ltv_at_default: Loan-to-value (LTV) at time of default
variables_df['ltv_at_default'] = (variables_df["upb_at_default"] * variables_df["original_loan_to_value"]) / variables_df["original_upb"]

# n_months: Age of loan in months at time of default
variables_df["date_default"] = pd.to_datetime(variables_df["date_default"])
variables_df["n_months"] = variables_df.apply(
    lambda r: relativedelta(r["date_default"], r["date_loan_start"]).years * 12 +
              relativedelta(r["date_default"], r["date_loan_start"]).months,
    axis=1
)

# 30y_avg_at_default: National average of 30y mortgage rates for year of default
rate_df["observation_date"] = pd.to_datetime(rate_df["observation_date"])
variables_df = variables_df.sort_values("date_default")
rate_df = rate_df.sort_values("observation_date")
variables_df = pd.merge_asof(
    variables_df,
    rate_df,
    left_on="date_default",
    right_on="observation_date",
    direction="backward"      # use the last rate at or before this date
)
variables_df = variables_df.rename(columns={"MORTGAGE30US": "30y_avg_at_default"})

# ir_spread: Difference between mortgage rate and 30Y national average at time of default
variables_df["ir_spread"] = variables_df["original_interest_rate"] - variables_df['30y_avg_at_default']

# Adding macro stats (by state) at time of default (taken year before default)
macro_df['year'] = pd.to_datetime(macro_df["year"] + "-12-31", format="%Y-%m-%d")
macro_df = macro_df.sort_values("year")

variables_df = pd.merge_asof(
    variables_df,
    macro_df,
    left_on="date_default",
    right_on="year",
    by="state_short",
    direction="backward"      # use the last rate at or before this date
)

In [ ]:
# Extracting feature matrix for regression
feature_cols = ["loan_number", "date_loan_start", "date_default", 'year_default', "X_centroid", "Y_centroid", 'ZIP3', "credit_score", "occupancy", "nr_units", "loan_purpose",
                "first_time_homebuyer", "MSA", "insurance_percent", "original_debt_to_income", "original_loan_to_value", "original_upb",
                "number_of_borrowers", "ir_spread", "n_months", "ltv_at_default", "gdp_growth", "ln_income_per_capita", "ln_expenditures_per_capita",
                "unemployment_rate", "hpi_growth", "gdp_construction_growth", "gross_operating_surplus_growth", "inflation_rate", "DJIA_growth", "payoff_progress",
                "LGD"]

features_df = variables_df[feature_cols]

features_df.to_feather("data/processed_data/LGD_dataset.feather")

# Versioning (Expanding window)

In [ ]:
import json
versioning_start = pd.Timestamp("2000-01-01")
versioning_dates = pd.date_range(versioning_start, periods=23, freq="YS")  # Year Start
prediction_lag = pd.DateOffset(years=1)

XY_all = []

version_dicts = []

for training_cutoff_date in versioning_dates:
    testing_cutoff_date = training_cutoff_date + prediction_lag

    new_testing_data = features_df[(features_df["date_default"] >= training_cutoff_date) & (features_df["date_default"] < testing_cutoff_date)].copy()

    training_data = pd.DataFrame(columns=feature_cols)
    if XY_all:
        training_data = pd.concat(XY_all, ignore_index=True)

    year = training_cutoff_date.year

    training_data.to_feather(f'data/processed_data/time_versioning/{year}_snapshot/training_data.feather')
    new_testing_data.to_feather(f'data/processed_data/time_versioning/{year}_snapshot/testing_data.feather')

    XY_all.append(new_testing_data)